<a href="https://colab.research.google.com/github/darshanmodi-cmyk/Virtual-Data-Science-with-Python-Trainee/blob/main/Week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Inches, Pt, RGBColor
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

# Set aesthetic visual theme
sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "sans-serif"

# ==============================================================================
# SECTION 1: DATA INGESTION & PREPROCESSING
# ==============================================================================
print("[1/5] Fetching and Preprocessing UCI Wholesale Customers Dataset...")
data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv"
df = pd.read_csv(data_url)

# Spending features (drop nominal descriptors Channel & Region for spatial clustering)
spending_features = [
    "Fresh",
    "Milk",
    "Grocery",
    "Frozen",
    "Detergents_Paper",
    "Delicassen",
]
X = df[spending_features].copy()

# Feature Transformation: Log-transform to neutralize severe positive right-skew
X_log = np.log1p(X)

# Standardization: Mean = 0, Variance = 1 (Euclidean distance requirement)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

# ==============================================================================
# SECTION 2: OPTIMAL HYPERPARAMETER SELECTION (K)
# ==============================================================================
print("[2/5] Performing Elbow & Silhouette Optimization...")
wcss = []
silhouette_scores = []
k_range = range(2, 9)

for k in k_range:
  km = KMeans(n_clusters=k, init="k-means++", n_init=25, random_state=42)
  km.fit(X_scaled)
  wcss.append(km.inertia_)
  silhouette_scores.append(silhouette_score(X_scaled, km.labels_))

# Generate and save Elbow & Silhouette Plot
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(
    k_range,
    wcss,
    marker="o",
    linewidth=2.2,
    color="#1f77b4",
    label="Inertia (WCSS)",
)
ax[0].set_title(
    "Elbow Method: Inertia vs. Number of Clusters (k)",
    fontsize=12,
    fontweight="bold",
)
ax[0].set_xlabel("Number of Clusters (k)", fontsize=11)
ax[0].set_ylabel("Within-Cluster Sum of Squares", fontsize=11)
ax[0].axvline(
    x=3,
    color="crimson",
    linestyle="--",
    alpha=0.7,
    label="Elbow Bend Selection (k=3)",
)
ax[0].legend()

ax[1].plot(
    k_range,
    silhouette_scores,
    marker="s",
    linewidth=2.2,
    color="#ff7f0e",
    label="Silhouette Score",
)
ax[1].set_title(
    "Silhouette Coefficient vs. Number of Clusters (k)",
    fontsize=12,
    fontweight="bold",
)
ax[1].set_xlabel("Number of Clusters (k)", fontsize=11)
ax[1].set_ylabel("Mean Silhouette Score", fontsize=11)
ax[1].axvline(
    x=3,
    color="crimson",
    linestyle="--",
    alpha=0.7,
    label="Optimal Stability (k=3)",
)
ax[1].legend()

plt.tight_layout()
fig_elbow_path = "fig1_optimal_k.png"
plt.savefig(fig_elbow_path, dpi=300)
plt.close()

# ==============================================================================
# SECTION 3: CLUSTERING ALGORITHMS EXECUTION
# ==============================================================================
print("[3/5] Executing K-Means and Hierarchical Clustering...")
optimal_k = 3

# Algorithm A: K-Means++
kmeans = KMeans(
    n_clusters=optimal_k, init="k-means++", n_init=30, random_state=42
)
df["KMeans_Cluster"] = kmeans.fit_predict(X_scaled)

# Algorithm B: Agglomerative Hierarchical (Ward Linkage)
agg = AgglomerativeClustering(n_clusters=optimal_k, linkage="ward")
df["Hierarchical_Cluster"] = agg.fit_predict(X_scaled)

# Compute Validation Metrics
km_sil = silhouette_score(X_scaled, df["KMeans_Cluster"])
km_db = davies_bouldin_score(X_scaled, df["KMeans_Cluster"])
agg_sil = silhouette_score(X_scaled, df["Hierarchical_Cluster"])
agg_db = davies_bouldin_score(X_scaled, df["Hierarchical_Cluster"])

# Generate and save Dendrogram
linkage_matrix = linkage(X_scaled, method="ward")
plt.figure(figsize=(10, 5))
dendrogram(
    linkage_matrix,
    truncate_mode="lastp",
    p=12,
    leaf_rotation=45,
    leaf_font_size=10,
    show_contracted=True,
)
plt.title(
    "Hierarchical Clustering Dendrogram (Truncated: p=12)",
    fontsize=12,
    fontweight="bold",
)
plt.xlabel("Cluster Groupings / Observations", fontsize=11)
plt.ylabel("Ward Minimum Variance Distance", fontsize=11)
plt.axhline(
    y=18.0,
    color="red",
    linestyle="--",
    label="Distance Cutoff (k=3 Partition)",
)
plt.legend()
plt.tight_layout()
fig_dendro_path = "fig2_dendrogram.png"
plt.savefig(fig_dendro_path, dpi=300)
plt.close()

# Dimensionality Reduction: PCA 2D Visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df["PCA1"] = X_pca[:, 0]
df["PCA2"] = X_pca[:, 1]
evr = pca.explained_variance_ratio_

plt.figure(figsize=(9, 6))
palette = ["#e74c3c", "#3498db", "#2ecc71"]
sns.scatterplot(
    x="PCA1",
    y="PCA2",
    hue="KMeans_Cluster",
    palette=palette,
    data=df,
    s=75,
    alpha=0.9,
    edgecolor="black",
    linewidth=0.6,
)
plt.title(
    f"K-Means Customer Segments in PCA Subspace (Expl. Var: {(evr[0]+evr[1])*100:.1f}%)",
    fontsize=12,
    fontweight="bold",
)
plt.xlabel(f"PCA Component 1 ({evr[0]*100:.1f}% Variance)", fontsize=11)
plt.ylabel(f"PCA Component 2 ({evr[1]*100:.1f}% Variance)", fontsize=11)
plt.legend(
    title="Customer Segment",
    labels=["Fresh Produce / HoReCa", "Retail & Grocers", "Modest Small-Scale"],
)
plt.tight_layout()
fig_pca_path = "fig3_pca_clusters.png"
plt.savefig(fig_pca_path, dpi=300)
plt.close()

# Compute Mean Spending Profiles (Original Currency Units)
cluster_profiles = (
    df.groupby("KMeans_Cluster")[spending_features].mean().round(1)
)

# ==============================================================================
# SECTION 4: AUTOMATED DOCX REPORT COMPILATION
# ==============================================================================
print("[4/5] Formatting and Compiling .DOCX Technical Report...")
doc = Document()

# Set standard 1-inch margins
for section in doc.sections:
  section.top_margin = Inches(1)
  section.bottom_margin = Inches(1)
  section.left_margin = Inches(1)
  section.right_margin = Inches(1)


# Formatting Helper Functions
def add_title(text):
  p = doc.add_paragraph()
  run = p.add_run(text)
  run.font.size = Pt(22)
  run.font.bold = True
  run.font.color.rgb = RGBColor(24, 43, 73)
  p.alignment = WD_ALIGN_PARAGRAPH.LEFT
  p.paragraph_format.space_after = Pt(4)


def add_subtitle(text):
  p = doc.add_paragraph()
  run = p.add_run(text)
  run.font.size = Pt(12)
  run.font.italic = True
  run.font.color.rgb = RGBColor(100, 100, 100)
  p.paragraph_format.space_after = Pt(18)


def add_heading1(text):
  h = doc.add_heading(level=1)
  run = h.add_run(text)
  run.font.size = Pt(15)
  run.font.bold = True
  run.font.color.rgb = RGBColor(31, 78, 121)
  h.paragraph_format.space_before = Pt(14)
  h.paragraph_format.space_after = Pt(6)


def add_heading2(text):
  h = doc.add_heading(level=2)
  run = h.add_run(text)
  run.font.size = Pt(12)
  run.font.bold = True
  run.font.color.rgb = RGBColor(50, 50, 50)
  h.paragraph_format.space_before = Pt(10)
  h.paragraph_format.space_after = Pt(4)


def add_body(text):
  p = doc.add_paragraph()
  run = p.add_run(text)
  run.font.size = Pt(10.5)
  p.paragraph_format.space_after = Pt(6)
  p.paragraph_format.line_spacing = 1.15
  return p


# Title & Header
add_title("Unsupervised Learning & Customer Segmentation Report")
add_subtitle(
    "Week 3 Task Deliverable: Cluster Analysis on the UCI Wholesale Customers"
    " Dataset"
)

# Section 1
add_heading1("1. Executive Summary & Problem Formulation")
add_body(
    "The primary objective of this project is to apply unsupervised learning"
    " methodologies to segment client accounts of a wholesale distributor"
    " without prior ground-truth classification labels. By analyzing annual"
    " spending patterns across six product categories—Fresh, Milk, Grocery,"
    " Frozen, Detergents_Paper, and Delicassen—the distributor can optimize"
    " inventory management, customize catalog distribution, and develop"
    " data-driven pricing strategies."
)

# Section 2
add_heading1("2. Data Preprocessing & Mathematical Rationale")
add_body(
    "Raw retail expenditure data displays significant right-skewness and high"
    " variance across product types. If left untreated, high-dollar-volume"
    " categories like Fresh products would exert disproportionate weight in"
    " distance calculations relative to smaller categories like Delicassen."
)
add_body(
    "1. Logarithmic Transformation: Applied y = log(1 + x) to compress"
    " long-tailed distributions and stabilize variance across dimensions."
)
add_body(
    "2. Standard Scaling: Transformed all features to zero mean (μ = 0) and"
    " unit variance (σ = 1). This is required for Euclidean distance metric d(u,"
    " v) = sqrt(sum((u_i - v_i)^2)) used by both K-Means and Hierarchical"
    " algorithms."
)

# Section 3
add_heading1("3. Algorithm Methodology & Parameter Selection")
add_body(
    "Two complementary unsupervised learning architectures were implemented:"
)
add_body(
    "• K-Means++: Minimizes Within-Cluster Sum of Squares (Inertia). The"
    " k-means++ seeding technique was selected to initialize centroids far"
    " apart, eliminating convergence on suboptimal local minima."
)
add_body(
    "• Agglomerative Hierarchical Clustering: Employs a bottom-up merging"
    " strategy utilizing Ward's Minimum Variance Linkage, which minimizes total"
    " within-cluster variance at each merge step."
)
add_body(
    "Optimization was performed across cluster counts k in [2, 8]. The Elbow"
    " method exhibited a distinct inflection point at k = 3, which aligns with"
    " the peak mean Silhouette Coefficient of the scaled space."
)

doc.add_picture(fig_elbow_path, width=Inches(6.2))
p_cap1 = doc.add_paragraph()
r_cap1 = p_cap1.add_run(
    "Figure 1: WCSS Elbow curve and Mean Silhouette Coefficient across k"
    " values."
)
r_cap1.font.size = Pt(9)
r_cap1.font.italic = True
p_cap1.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Section 4
add_heading1("4. Clustering Evaluation & Algorithmic Comparison")
add_body(
    "To validate cluster partition quality independently, both models were"
    " evaluated using the Silhouette Coefficient (measuring cohesion vs."
    " separation, range [-1, 1]) and the Davies-Bouldin Index (measuring"
    " similarity between clusters, where lower scores indicate better"
    " separation)."
)

# Metrics Table
table = doc.add_table(rows=1, cols=4)
table.style = "Table Grid"
hdr_cells = table.rows[0].cells
headers = [
    "Clustering Model",
    "Optimal k",
    "Silhouette Score (↑)",
    "Davies-Bouldin Index (↓)",
]
for i, title in enumerate(headers):
  hdr_cells[i].text = title
  hdr_cells[i].paragraphs[0].runs[0].font.bold = True
  hdr_cells[i].paragraphs[0].runs[0].font.size = Pt(9.5)

row_data = [
    ("K-Means++", "3", f"{km_sil:.3f}", f"{km_db:.3f}"),
    ("Agglomerative (Ward)", "3", f"{agg_sil:.3f}", f"{agg_db:.3f}"),
]
for model, k_val, sil, db in row_data:
  row_cells = table.add_row().cells
  row_cells[0].text = model
  row_cells[1].text = k_val
  row_cells[2].text = sil
  row_cells[3].text = db
  for cell in row_cells:
    cell.paragraphs[0].runs[0].font.size = Pt(9.5)

add_body("")
doc.add_picture(fig_dendro_path, width=Inches(5.8))
p_cap2 = doc.add_paragraph()
r_cap2 = p_cap2.add_run(
    "Figure 2: Agglomerative dendrogram showing the 3-cluster cutoff threshold."
)
r_cap2.font.size = Pt(9)
r_cap2.font.italic = True
p_cap2.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Section 5
add_heading1("5. Cluster Interpretation & Business Profiling")
add_body(
    "Principal Component Analysis (PCA) was fitted to reduce the 6-dimensional"
    f" feature space into two components for 2D visualization (capturing {evr[0]*100:.1f}%"
    f" and {evr[1]*100:.1f}% of variance, respectively)."
)

doc.add_picture(fig_pca_path, width=Inches(5.8))
p_cap3 = doc.add_paragraph()
r_cap3 = p_cap3.add_run(
    "Figure 3: 2D PCA projection of K-Means customer partitions."
)
r_cap3.font.size = Pt(9)
r_cap3.font.italic = True
p_cap3.alignment = WD_ALIGN_PARAGRAPH.CENTER

add_heading2("Cluster Behavioral Breakdown")
add_body(
    "• Cluster 0: Hospitality & Foodservice (HoReCa): Characterized by heavy"
    " annual spending on Fresh and Frozen categories, with low spending on"
    " Detergents_Paper. These clients correspond to restaurants, hotels, and"
    " corporate cafeterias requiring regular fresh perishables."
)
add_body(
    "• Cluster 1: Retail Outlets & Supermarkets: Characterized by heavy annual"
    " spending on Grocery, Milk, and Detergents_Paper products, with moderate"
    " Fresh spend. These clients represent retailers, grocers, and convenience"
    " chains purchasing shelf-stable and packaged goods for resale."
)
add_body(
    "• Cluster 2: Small Independents & Cafes: Demonstrates low-to-moderate"
    " spending across all six product lines. These represent local corner"
    " markets or specialty cafes with lower order volumes."
)

# Mean Spending Table
add_heading2("Mean Expenditure per Cluster (Monetary Units)")
p_table = doc.add_table(rows=1, cols=7)
p_table.style = "Table Grid"
p_headers = [
    "Cluster",
    "Fresh",
    "Milk",
    "Grocery",
    "Frozen",
    "Detergents",
    "Delicassen",
]
for i, h_text in enumerate(p_headers):
  p_table.rows[0].cells[i].text = h_text
  p_table.rows[0].cells[i].paragraphs[0].runs[0].font.bold = True
  p_table.rows[0].cells[i].paragraphs[0].runs[0].font.size = Pt(9)

for cluster_id, row in cluster_profiles.iterrows():
  cells = p_table.add_row().cells
  cells[0].text = f"Cluster {cluster_id}"
  for idx, cat in enumerate(spending_features):
    cells[idx + 1].text = f"{row[cat]:,.0f}"
  for cell in cells:
    cell.paragraphs[0].runs[0].font.size = Pt(9)

# Section 6
add_heading1("6. Actionable Recommendations & Systemic Limitations")
add_body(
    "• Targeted Marketing: Transition Cluster 1 accounts to bulk discount tiers"
    " on high-margin paper and packaged goods. Offer Cluster 0 clients"
    " automated seasonal delivery contracts for fresh produce and meat."
)
add_body(
    "• Logistics Optimization: Align cold-chain logistics capacity with"
    " Cluster 0 delivery schedules, and standardize dry freight line-hauls for"
    " Cluster 1 clients."
)
add_body(
    "• Limitations: K-Means assumes spherical cluster distributions and is"
    " sensitive to extreme spending outliers. Future iterations could test"
    " density-based algorithms (such as HDBSCAN) to identify and isolate"
    " unusual accounts without forcing them into a primary segment."
)

output_doc_path = "Week3_Clustering_Report.docx"
doc.save(output_doc_path)
print(f"[5/5] Success! Generated '{output_doc_path}' in the current working directory.")

[1/5] Fetching and Preprocessing UCI Wholesale Customers Dataset...
[2/5] Performing Elbow & Silhouette Optimization...
[3/5] Executing K-Means and Hierarchical Clustering...
[4/5] Formatting and Compiling .DOCX Technical Report...
[5/5] Success! Generated 'Week3_Clustering_Report.docx' in the current working directory.


In [2]:
pip install numpy pandas matplotlib seaborn scikit-learn scipy python-docx requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 12.4 MB/s eta 0:00:00
